# 1.1b Vector stores in depth: Amazon S3 Vectors and Amazon OpenSearch Serverless

**Story:** the SafeDrive claims assistant must find the right policy paragraph, similar past claim
notes and similar damage photos. All three are *similarity searches* over embeddings, so you need a
vector store. Here you load the same data into **S3 Vectors** and **OpenSearch Serverless**, search
it in different ways, and learn when to choose which.

**Task 1.1** Collect and store data.

| Skill | What | Status |
|---|---|---|
| 1.1.7 | Configure scalable vector databases (S3 Vectors, OpenSearch; pgvector as notes) | ✅ |
| 1.1.8 | Store diverse data types as vectors (text, short notes, images) | ✅ |

💰 **Cost:** Part A (S3 Vectors) costs cents. Part B (OpenSearch Serverless) bills about
USD 0.25-0.50 **per hour while the collection exists**. The cleanup section deletes it - do not skip it.
If you do not want that cost, read Part B without running it.

## Key points for the exam

- A vector store holds **embeddings** and returns the **k nearest** vectors to a query vector.
- Two settings must match the embedding model: **dimension** and **distance metric**. Both are fixed
  when the index is created. To change them you build a new index and reload.
- **Metadata** stored next to a vector lets you **filter** (for example only `category = claims`).
- **S3 Vectors** = cheapest, no servers, huge scale, semantic search only, sub-second latency.
- **OpenSearch** = lowest latency, high query rates, **hybrid search** (keywords + vectors),
  aggregations, rich filters. You pay for compute all the time.
- **Aurora / RDS PostgreSQL with pgvector** = vectors next to relational data, SQL joins.
- **Is Bedrock Knowledge Bases on the exam?** Yes: skill 3.2.7 (create and manage knowledge bases),
  3.1.8 and 3.2.8 (retrieval). A Knowledge Base *uses* one of these vector stores underneath and does
  the chunking, embedding and loading for you. This notebook does that work by hand so you understand
  it; notebook **3.2b** builds the Knowledge Base.

| Need | Choose |
|---|---|
| Millions to billions of vectors, queried now and then, lowest cost | **S3 Vectors** |
| Interactive app, many queries per second, tens of milliseconds | **OpenSearch** |
| Must match exact words too (policy numbers, error codes, names) | **OpenSearch** (hybrid search) |
| Facets, counts, full-text relevance tuning | **OpenSearch** |
| Data already in PostgreSQL, need joins and transactions | **Aurora / RDS pgvector** |
| Answers need relationships between entities (GraphRAG) | **Neptune Analytics** |
| Hot data fast, cold data cheap | S3 Vectors for storage + OpenSearch for the hot subset |

## Setup

In [ ]:
import sys; sys.path.append("..")
import base64, json, time
import numpy as np
import pandas as pd
from common import utils as U, catchup as C
print(U.REGION, U.bucket())

Catch-up: raw data, and the policy chunks with their 1024-dimension embeddings (built in 1.2b).

In [ ]:
C.ensure_raw_data()
chunks = C.ensure_chunks()
print(chunks.shape)
chunks[["source", "heading", "text"]].head(3)

Clients used in this notebook.

In [ ]:
s3v = U.client("s3vectors")
brt = U.client("bedrock-runtime")
aoss = U.client("opensearchserverless")
sts = U.client("sts")

## Prepare three kinds of vectors

| Data | Embedding model | Dimension | Why |
|---|---|---|---|
| Policy chunks (long text) | Titan Text Embeddings V2 | 1024 | Best quality for question answering |
| Adjuster notes (short text) | Titan Text Embeddings V2 | 256 | Smaller = cheaper storage, faster search, slightly less accurate |
| Damage photos (images) | Titan Multimodal Embeddings | 384 | Text and images share one vector space |

### Vectors 1: policy chunks (already embedded)
Add a `category` to each chunk, taken from the file name. It becomes filterable metadata.

In [ ]:
CATEGORY = {"01": "coverage", "02": "exclusions", "03": "claims", "04": "fraud", "05": "pricing"}
chunks["category"] = chunks["source"].str[:2].map(CATEGORY)
chunks["key"] = [f"chunk-{i:03d}" for i in range(len(chunks))]
chunks[["key", "category", "heading"]].head()

### Vectors 2: adjuster notes at 256 dimensions
Load 120 notes.

In [ ]:
notes = pd.DataFrame(U.read_jsonl(f"{C.RAW}/adjuster_notes.jsonl")).head(120)
notes[["claim_id", "claim_type", "note"]].head(3)

Embed them with `dimensions=256` (one API call per note, about a minute).

In [ ]:
notes["embedding"] = [U.embed(t, dimensions=256) for t in notes["note"]]
print(len(notes), "notes, vector length", len(notes["embedding"][0]))

### Vectors 3: damage photos with the multimodal model
The multimodal model takes an image **or** a text and puts both in the same space.

In [ ]:
def embed_multimodal(text=None, image_bytes=None, dim=384):
    body = {"embeddingConfig": {"outputEmbeddingLength": dim}}
    if text:
        body["inputText"] = text
    if image_bytes:
        body["inputImage"] = base64.b64encode(image_bytes).decode()
    resp = brt.invoke_model(modelId=U.IMAGE_EMBED_MODEL, body=json.dumps(body))
    return json.loads(resp["body"].read())["embedding"]

Load the image manifest (file name, colour, damage type) and embed each image.

In [ ]:
images = pd.DataFrame(U.read_jsonl(f"{C.RAW}/images/manifest.jsonl"))
images["embedding"] = [embed_multimodal(image_bytes=U.read_bytes(f"{C.RAW}/images/{f}")) for f in images["file"]]
print(len(images), "images, vector length", len(images["embedding"][0]))

---
# Part A: Amazon S3 Vectors

- **Vector bucket** → **vector index** → **vectors** (key + data + metadata).
- No cluster to size. You pay for storage, for PUTs and for queries.
- An index has a fixed **dimension**, **distance metric** (cosine or Euclidean) and list of
  **non-filterable** metadata keys.
- Metadata is **filterable by default**. Declare big fields (the chunk text) as non-filterable: they
  are returned with results but cannot be used in a filter, and they have a larger size limit.

### A1. Create (or reuse) the vector bucket
This is the same vector bucket the Knowledge Base uses later; we add our own lab indexes to it.

In [ ]:
VBUCKET = f"safedrive-vectors-{U.account_id()}"
try:
    s3v.get_vector_bucket(vectorBucketName=VBUCKET)
    print("vector bucket exists")
except s3v.exceptions.NotFoundException:
    s3v.create_vector_bucket(vectorBucketName=VBUCKET)
    print("vector bucket created")

### A2. Create three indexes, one per kind of vector
A small helper that creates an index only if it is missing.

In [ ]:
def create_index(name, dimension, metric, non_filterable=None):
    existing = [i["indexName"] for i in s3v.list_indexes(vectorBucketName=VBUCKET)["indexes"]]
    if name in existing:
        return print("exists:", name)
    args = dict(vectorBucketName=VBUCKET, indexName=name, dataType="float32",
                dimension=dimension, distanceMetric=metric)
    if non_filterable:
        args["metadataConfiguration"] = {"nonFilterableMetadataKeys": non_filterable}
    s3v.create_index(**args)
    print("created:", name)

Policy chunks: 1024 dimensions, cosine. The chunk `text` is non-filterable.

In [ ]:
create_index("lab-policy-1024", 1024, "cosine", non_filterable=["text"])

Notes: 256 dimensions, Euclidean (to see the other metric). Photos: 384 dimensions, cosine.

In [ ]:
create_index("lab-notes-256", 256, "euclidean", non_filterable=["note"])
create_index("lab-images-384", 384, "cosine")

Look at one index definition. These settings cannot be changed later.

In [ ]:
idx = s3v.get_index(vectorBucketName=VBUCKET, indexName="lab-policy-1024")["index"]
print(idx["dimension"], idx["distanceMetric"], idx.get("metadataConfiguration"))

### A3. Load vectors with `put_vectors`
One vector = `key` + `data` (list of float32) + `metadata`. Writing the same key again overwrites it.

In [ ]:
policy_vectors = [
    {"key": r["key"], "data": {"float32": [float(x) for x in r["embedding"]]},
     "metadata": {"category": r["category"], "source": r["source"], "heading": r["heading"], "text": r["text"]}}
    for r in chunks.to_dict("records")]
print(len(policy_vectors), "vectors;", policy_vectors[0]["key"], list(policy_vectors[0]["metadata"]))

Send them in one call (a call takes up to a few hundred vectors; batch larger loads).

In [ ]:
s3v.put_vectors(vectorBucketName=VBUCKET, indexName="lab-policy-1024", vectors=policy_vectors)
print("loaded policy chunks")

Notes: metadata can be a string, number, boolean or list. `is_fraud` is stored as a number.

In [ ]:
note_vectors = [
    {"key": r["claim_id"], "data": {"float32": [float(x) for x in r["embedding"]]},
     "metadata": {"claim_type": r["claim_type"], "is_fraud": int(r["is_fraud"]), "note": r["note"]}}
    for r in notes.to_dict("records")]

Load in batches of 50 - the pattern you need for large data sets.

In [ ]:
for start in range(0, len(note_vectors), 50):
    batch = note_vectors[start:start + 50]
    s3v.put_vectors(vectorBucketName=VBUCKET, indexName="lab-notes-256", vectors=batch)
    print("loaded", start + len(batch))

Photos.

In [ ]:
image_vectors = [
    {"key": r["file"], "data": {"float32": [float(x) for x in r["embedding"]]},
     "metadata": {"damage_type": r["damage_type"], "vehicle_color": r["vehicle_color"], "claim_id": r["claim_id"]}}
    for r in images.to_dict("records")]
s3v.put_vectors(vectorBucketName=VBUCKET, indexName="lab-images-384", vectors=image_vectors)
print("loaded", len(image_vectors), "images")

### A4. Read vectors back
`list_vectors` pages through an index; `get_vectors` fetches by key.

In [ ]:
listed = s3v.list_vectors(vectorBucketName=VBUCKET, indexName="lab-policy-1024", maxResults=5)
[v["key"] for v in listed["vectors"]]

In [ ]:
got = s3v.get_vectors(vectorBucketName=VBUCKET, indexName="lab-policy-1024",
                      keys=["chunk-000"], returnMetadata=True, returnData=True)
v = got["vectors"][0]
print(v["key"], "| dims:", len(v["data"]["float32"]), "|", v["metadata"]["heading"])

### A5. Similarity search
Embed the question with the **same model and dimension** as the index, then ask for the top K.
`distance`: smaller = more similar.

In [ ]:
def s3v_search(index, vector, k=3, flt=None):
    args = dict(vectorBucketName=VBUCKET, indexName=index, topK=k,
                queryVector={"float32": [float(x) for x in vector]},
                returnMetadata=True, returnDistance=True)
    if flt:
        args["filter"] = flt
    return s3v.query_vectors(**args)["vectors"]

In [ ]:
QUESTION = "How much do I pay myself when I make a claim?"
q1024 = U.embed(QUESTION)
for v in s3v_search("lab-policy-1024", q1024):
    print(round(v["distance"], 3), "|", v["metadata"]["category"], "|", v["metadata"]["heading"])

Note that the question says "pay myself" and the best chunk says "deductible": semantic search
matches **meaning**, not words.

### A6. Metadata filters
The filter is applied **during** the search, so you still get K results from the matching subset.

| Operator | Meaning |
|---|---|
| `$eq`, `$ne` | equals / not equals |
| `$gt`, `$gte`, `$lt`, `$lte` | number comparisons |
| `$in`, `$nin` | value in / not in a list |
| `$exists` | the key is present |
| `$and`, `$or` | combine conditions |

Only chunks from the *exclusions* document.

In [ ]:
only_exclusions = {"category": {"$eq": "exclusions"}}
for v in s3v_search("lab-policy-1024", q1024, flt=only_exclusions):
    print(round(v["distance"], 3), "|", v["metadata"]["category"], "|", v["metadata"]["heading"])

Notes similar to a suspicious story, restricted to theft or fire claims flagged as fraud.

In [ ]:
q256 = U.embed("customer changed the story and there is no police report", dimensions=256)
suspicious = {"$and": [{"claim_type": {"$in": ["theft", "fire"]}}, {"is_fraud": {"$eq": 1}}]}
for v in s3v_search("lab-notes-256", q256, k=5, flt=suspicious):
    print(round(v["distance"], 3), "|", v["key"], v["metadata"]["claim_type"], "|", v["metadata"]["note"][:70])

Filtering on a **non-filterable** key fails. Run this to see the error you would get.

In [ ]:
try:
    s3v_search("lab-policy-1024", q1024, flt={"text": {"$eq": "anything"}})
except Exception as e:
    print(type(e).__name__, "-", str(e)[:160])

### A7. A wrong dimension is rejected
A 256-dimension query cannot be used on a 1024-dimension index.

In [ ]:
try:
    s3v_search("lab-policy-1024", q256)
except Exception as e:
    print(type(e).__name__, "-", str(e)[:160])

### A8. Image search with a text query (multimodal)
Embed the *text* with the multimodal model and search the *image* index.

In [ ]:
q_img = embed_multimodal(text="a car with a broken window")
for v in s3v_search("lab-images-384", q_img, k=4):
    print(round(v["distance"], 3), "|", v["key"], v["metadata"]["damage_type"], v["metadata"]["vehicle_color"])

> The pictures are simple drawings, so the ranking may not look meaningful. The point is the
> mechanics: text query → multimodal embedding → image vectors.

Image-to-image: find photos similar to one photo ("have we seen this damage before?").

In [ ]:
for v in s3v_search("lab-images-384", images["embedding"][0], k=3):
    print(round(v["distance"], 3), "|", v["key"], v["metadata"]["damage_type"])

### A9. Update and delete
Update = `put_vectors` with the same key. Delete by key.

In [ ]:
s3v.delete_vectors(vectorBucketName=VBUCKET, indexName="lab-notes-256", keys=[notes["claim_id"][0]])
remaining = s3v.get_vectors(vectorBucketName=VBUCKET, indexName="lab-notes-256", keys=[notes["claim_id"][0]])
print("still there:", len(remaining["vectors"]))

### S3 Vectors: what to remember
- Semantic (vector) search only. **No keyword or hybrid search**, no aggregations.
- Sub-second queries; built for large, cost-sensitive, less frequently queried data.
- Limits exist on `topK`, vectors per PUT and metadata size - check the quotas page for current values.
- Access is controlled with IAM (`s3vectors:*` actions); encryption with SSE-S3 or KMS.
- Works as a **Bedrock Knowledge Base** vector store and can export to OpenSearch for hot data.

---
# Part B: Amazon OpenSearch Serverless

💰 Billing starts when the collection is created and stops when it is deleted.

- OpenSearch is a **search engine**: full-text (BM25), filters, aggregations **and** k-NN vectors.
- Two ways to run it: **Serverless** (collections, capacity in OCUs, no nodes to manage) or a
  **managed domain** (you choose instances; more engines and tuning options).
- Serverless hierarchy: **collection** (type `VECTORSEARCH`) → **index** → **documents**.
- A collection needs three policies: **encryption**, **network**, **data access**.
- Vectors live in a field of type `knn_vector`, next to normal text and keyword fields.

### B1. Who am I? (for the data access policy)
Data access policies name IAM principals. We use the identity that runs this notebook.

In [ ]:
caller = sts.get_caller_identity()["Arn"]
if ":assumed-role/" in caller:
    role_name = caller.split("/")[1]
    caller = f"arn:aws:iam::{U.account_id()}:role/{role_name}"
print(caller)

> If your role has a path (for example `/service-role/`), put the full role ARN here by hand.
> Your identity also needs the IAM permission `aoss:APIAccessAll`.

### B2. Encryption policy
Required before the collection can be created. Here: an AWS owned key.

In [ ]:
COLL = "safedrive-vector-lab"
enc_policy = {"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLL}"]}], "AWSOwnedKey": True}
U.safe(aoss.create_security_policy, name=f"{COLL}-enc", type="encryption", policy=json.dumps(enc_policy))

### B3. Network policy
Public endpoint for this lab (requests are still signed with IAM). Production: VPC endpoint only.

In [ ]:
net_policy = [{"Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLL}"]}], "AllowFromPublic": True}]
U.safe(aoss.create_security_policy, name=f"{COLL}-net", type="network", policy=json.dumps(net_policy))

### B4. Data access policy
Who may create indexes and read/write documents. This is separate from IAM policies.

In [ ]:
data_policy = [{
    "Rules": [{"ResourceType": "collection", "Resource": [f"collection/{COLL}"], "Permission": ["aoss:*"]},
              {"ResourceType": "index", "Resource": [f"index/{COLL}/*"], "Permission": ["aoss:*"]}],
    "Principal": [caller, U.role_arn()]}]
U.safe(aoss.create_access_policy, name=f"{COLL}-data", type="data", policy=json.dumps(data_policy))

### B5. Create the vector search collection
`standbyReplicas="DISABLED"` halves the minimum capacity (fine for dev/test, not for production).

In [ ]:
U.safe(aoss.create_collection, name=COLL, type="VECTORSEARCH", standbyReplicas="DISABLED",
       tags=[{"key": t["Key"], "value": t["Value"]} for t in U.TAGS])

Wait until it is `ACTIVE` (a few minutes).

In [ ]:
get_coll = lambda: aoss.batch_get_collection(names=[COLL])["collectionDetails"][0]  # noqa: E731
U.wait_until(lambda: get_coll()["status"], ok=("ACTIVE",), bad=("FAILED",), delay=20, label="collection")

In [ ]:
endpoint = get_coll()["collectionEndpoint"]
print(endpoint)

### B6. Connect with `opensearch-py`
Requests are signed with SigV4 for the service name **`aoss`** (a managed domain uses `es`).

In [ ]:
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth, helpers
auth = AWSV4SignerAuth(U.session.get_credentials(), U.REGION, "aoss")
os_client = OpenSearch(hosts=[{"host": endpoint.replace("https://", ""), "port": 443}],
                       http_auth=auth, use_ssl=True, verify_certs=True,
                       connection_class=RequestsHttpConnection, timeout=60)

### B7. Create a vector index for the policy chunks

| Mapping part | Meaning |
|---|---|
| `index.knn: true` | turn on k-NN for this index |
| `knn_vector` + `dimension` | the vector field; must equal the embedding size |
| `method.name` | `hnsw` (graph: fast, accurate, more memory) or `ivf` (clusters: less memory, needs training) |
| `space_type` | `l2` (Euclidean), `cosinesimil`, `innerproduct` |
| `text` field | analysed words for keyword (BM25) search |
| `keyword` field | exact values for filters and aggregations |

Titan vectors are normalized (length 1), so `l2` gives the same ranking as cosine.

In [ ]:
policy_mapping = {
    "settings": {"index.knn": True},
    "mappings": {"properties": {
        "embedding": {"type": "knn_vector", "dimension": 1024,
                      "method": {"name": "hnsw", "engine": "faiss", "space_type": "l2",
                                 "parameters": {"m": 16, "ef_construction": 128}}},
        "text": {"type": "text"}, "heading": {"type": "text"},
        "category": {"type": "keyword"}, "source": {"type": "keyword"}}}}

Create it. If you get a 403 here, the data access policy has not propagated yet: wait a minute and re-run.

In [ ]:
if not os_client.indices.exists(index="policy-chunks"):
    os_client.indices.create(index="policy-chunks", body=policy_mapping)
print(os_client.indices.get_mapping(index="policy-chunks")["policy-chunks"]["mappings"]["properties"].keys())

HNSW parameters (exam level): `m` = links per node, `ef_construction` = build effort, `ef_search` =
query effort. Higher values = better recall, more memory and time.

### B8. Bulk-load the chunks
One document = the vector **plus** the text and metadata fields.
In a *vector search* collection you cannot choose the document `_id`; OpenSearch generates it.

In [ ]:
def to_actions(index, records, fields):
    return [{"_index": index, "_source": {"embedding": [float(x) for x in r["embedding"]],
                                          **{f: r[f] for f in fields}}} for r in records]

In [ ]:
actions = to_actions("policy-chunks", chunks.to_dict("records"), ["text", "heading", "category", "source"])
ok, errors = helpers.bulk(os_client, actions)
print("indexed:", ok, "errors:", errors)

Serverless makes new documents searchable after a short delay (there is no manual refresh).
Wait until the count matches.

In [ ]:
def wait_for_count(index, expected, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        n = os_client.count(index=index)["count"]
        print("searchable documents:", n)
        if n >= expected:
            return n
        time.sleep(15)

wait_for_count("policy-chunks", len(chunks))

A helper to print search hits. `_score`: **larger = more relevant** (the opposite of a distance).

In [ ]:
def show(resp, fields=("category", "heading")):
    for h in resp["hits"]["hits"]:
        print(round(h["_score"], 3), "|", " | ".join(str(h["_source"].get(f))[:60] for f in fields))

### B9. Vector (k-NN) search
The same question as in Part A.

In [ ]:
knn_query = {"size": 3, "_source": ["category", "heading"],
             "query": {"knn": {"embedding": {"vector": q1024, "k": 3}}}}
show(os_client.search(index="policy-chunks", body=knn_query))

### B10. Vector search with a filter
Putting `filter` **inside** the `knn` clause filters during the search (efficient filtering), so you
still get `k` results. A filter applied after the search (post-filter) can return fewer than `k`.

In [ ]:
filtered_query = {"size": 3, "_source": ["category", "heading"],
                  "query": {"knn": {"embedding": {"vector": q1024, "k": 3,
                                                  "filter": {"term": {"category": "exclusions"}}}}}}
show(os_client.search(index="policy-chunks", body=filtered_query))

### B11. Keyword (BM25) search - what S3 Vectors cannot do
Exact words matter here: "NCB" is an abbreviation that an embedding may not place well.

In [ ]:
KEYWORD_Q = "NCB Protect add-on"
bm25_query = {"size": 3, "_source": ["category", "heading"], "query": {"match": {"text": KEYWORD_Q}}}
show(os_client.search(index="policy-chunks", body=bm25_query))

The same words as a pure vector search, to compare.

In [ ]:
kw_vec = U.embed(KEYWORD_Q)
show(os_client.search(index="policy-chunks", body={
    "size": 3, "_source": ["category", "heading"], "query": {"knn": {"embedding": {"vector": kw_vec, "k": 3}}}}))

### B12. Hybrid search: keywords + vectors
Run both searches and merge the two rankings with **reciprocal rank fusion (RRF)**:
a document scores `1 / (60 + rank)` in each list, and the scores are added.

In [ ]:
def ranked_ids(body, size=10):
    hits = os_client.search(index="policy-chunks", body={**body, "size": size, "_source": ["heading"]})["hits"]["hits"]
    return [(h["_id"], h["_source"]["heading"]) for h in hits]

def hybrid(question, k=3):
    lists = [ranked_ids({"query": {"match": {"text": question}}}),
             ranked_ids({"query": {"knn": {"embedding": {"vector": U.embed(question), "k": 10}}}})]
    scores, names = {}, {}
    for hits in lists:
        for rank, (doc_id, heading) in enumerate(hits):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (60 + rank + 1)
            names[doc_id] = heading
    return [(round(s, 4), names[d]) for d, s in sorted(scores.items(), key=lambda x: -x[1])[:k]]

In [ ]:
hybrid("Does the NCB Protect add-on keep my discount after an accident?")

> OpenSearch can also do this in one request: a `hybrid` query plus a **search pipeline** with a
> normalization (or RRF) processor. Bedrock Knowledge Bases call this `HYBRID` search type, and it
> needs a vector store that supports it - OpenSearch does, S3 Vectors does not.

### B13. Aggregations and filters in one query
How many chunks per category? Search engines answer this directly; pure vector stores do not.

In [ ]:
agg = os_client.search(index="policy-chunks", body={
    "size": 0, "aggs": {"by_category": {"terms": {"field": "category"}}}})
{b["key"]: b["doc_count"] for b in agg["aggregations"]["by_category"]["buckets"]}

### B14. A second index with a different vector type: notes at 256 dimensions
A different dimension needs its own index (or its own field).

In [ ]:
notes_mapping = {
    "settings": {"index.knn": True},
    "mappings": {"properties": {
        "embedding": {"type": "knn_vector", "dimension": 256,
                      "method": {"name": "hnsw", "engine": "faiss", "space_type": "l2"}},
        "note": {"type": "text"}, "claim_id": {"type": "keyword"},
        "claim_type": {"type": "keyword"}, "is_fraud": {"type": "integer"}}}}
if not os_client.indices.exists(index="claim-notes"):
    os_client.indices.create(index="claim-notes", body=notes_mapping)

In [ ]:
note_actions = to_actions("claim-notes", notes.to_dict("records"), ["note", "claim_id", "claim_type", "is_fraud"])
ok, errors = helpers.bulk(os_client, note_actions)
print("indexed:", ok, "errors:", errors)

In [ ]:
wait_for_count("claim-notes", len(notes))

Similar notes, only fraud cases, using a `bool` filter with two conditions.

In [ ]:
note_filter = {"bool": {"must": [{"term": {"is_fraud": 1}}, {"terms": {"claim_type": ["theft", "fire"]}}]}}
note_query = {"size": 5, "_source": ["claim_id", "claim_type", "note"],
              "query": {"knn": {"embedding": {"vector": q256, "k": 5, "filter": note_filter}}}}
show(os_client.search(index="claim-notes", body=note_query), fields=("claim_id", "claim_type", "note"))

### B15. A third index: image vectors (384 dimensions)

In [ ]:
image_mapping = {
    "settings": {"index.knn": True},
    "mappings": {"properties": {
        "embedding": {"type": "knn_vector", "dimension": 384,
                      "method": {"name": "hnsw", "engine": "faiss", "space_type": "l2"}},
        "file": {"type": "keyword"}, "damage_type": {"type": "keyword"}, "vehicle_color": {"type": "keyword"}}}}
if not os_client.indices.exists(index="damage-images"):
    os_client.indices.create(index="damage-images", body=image_mapping)

In [ ]:
image_actions = to_actions("damage-images", images.to_dict("records"), ["file", "damage_type", "vehicle_color"])
ok, errors = helpers.bulk(os_client, image_actions)
wait_for_count("damage-images", len(images))

Text-to-image search, as in A8.

In [ ]:
img_query = {"size": 4, "_source": ["file", "damage_type", "vehicle_color"],
             "query": {"knn": {"embedding": {"vector": q_img, "k": 4}}}}
show(os_client.search(index="damage-images", body=img_query), fields=("file", "damage_type", "vehicle_color"))

### More vector types in OpenSearch (notes)

| Type | What it is | Why |
|---|---|---|
| `float` (default) | 32-bit floats, 4 bytes per dimension | Best accuracy |
| `byte` (`"data_type": "byte"`) | values -128 to 127, 1 byte per dimension | About 4x less memory; you quantize the vectors first |
| `binary` (`"data_type": "binary"`) | 1 bit per dimension, Hamming distance | About 32x less memory; needs a model that outputs binary embeddings (Titan Text Embeddings V2 can - see notebook 1.2b) |
| Built-in quantization (FP16, product quantization, disk-based mode) | OpenSearch compresses for you | Lower memory at a small recall cost |

Engine and algorithm choices: **HNSW** (default, fast, memory hungry), **IVF** (less memory, needs
a training step), **exact k-NN** with a script (perfect recall, slow, fine for small filtered sets).
Support differs between Serverless and managed domains and between versions - check the docs before
relying on one.

### OpenSearch: what to remember
- Use it when you need **low latency, high query volume, hybrid search, filters and aggregations**.
- Serverless capacity is measured in **OCUs** (separate for indexing and search) and has a minimum
  that is billed even when idle. Collection types: `SEARCH`, `TIMESERIES`, `VECTORSEARCH`.
- Three policies: encryption (KMS), network (public or VPC endpoint), data access (who can do what).
- A managed **domain** gives more control (instance types, more engines, fine tuning, UltraWarm).
- New documents are searchable after a short delay, not at once.

---
# Part C: Side by side

Ask both stores the same question and time one query each (one sample, so only a rough feel).

In [ ]:
t0 = time.time()
a = s3v_search("lab-policy-1024", q1024, k=3)
t1 = time.time()
b = os_client.search(index="policy-chunks", body=knn_query)
t2 = time.time()
print("S3 Vectors :", round((t1 - t0) * 1000), "ms |", [v["metadata"]["heading"] for v in a])
print("OpenSearch :", round((t2 - t1) * 1000), "ms |", [h["_source"]["heading"] for h in b["hits"]["hits"]])

| | S3 Vectors | OpenSearch Serverless | Aurora / RDS pgvector |
|---|---|---|---|
| What it is | Vector storage in S3 | Search engine with k-NN | Relational database with a vector type |
| Pay for | Storage + requests | OCUs per hour (minimum even when idle) | Database instance or ACUs |
| Latency | Sub-second | Milliseconds | Milliseconds (depends on index and size) |
| Keyword / hybrid search | No | **Yes** | Possible with PostgreSQL full-text, manual |
| Filters | Metadata operators | Full query DSL | SQL `WHERE`, joins |
| Aggregations | No | Yes | SQL |
| Operations | None | Low (policies, capacity limits) | Database administration |
| Bedrock Knowledge Base store | Yes | Yes | Yes (Aurora) |

**pgvector in four lines** (not run here):

```sql
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE policy_chunks (id bigserial PRIMARY KEY, category text, chunk text, embedding vector(1024));
CREATE INDEX ON policy_chunks USING hnsw (embedding vector_cosine_ops);
SELECT chunk FROM policy_chunks WHERE category = 'claims' ORDER BY embedding <=> :query_vector LIMIT 3;
```

**Where Bedrock Knowledge Bases fit:** a Knowledge Base automates what you just did by hand - read
documents from S3, chunk, embed, write to one of these stores, and query it with `Retrieve` or
`RetrieveAndGenerate`. You still choose the store, the embedding model and the dimension, so the
trade-offs above are exactly what the exam asks about. See notebook 3.2b.

## Exam traps
- Dimension and distance metric are fixed at index creation: changing the embedding model means **re-indexing**.
- "Lowest cost for a very large, rarely queried vector set" → S3 Vectors. "Hybrid search" or "lowest latency" → OpenSearch.
- OpenSearch Serverless needs an **encryption policy before** the collection, and a **data access policy** in addition to IAM.
- A filter inside the k-NN search keeps K results; a post-filter may return fewer.
- Query and documents must use the **same embedding model**; text and images can share a space only with a **multimodal** model.

## Quiz

**1.** A team stores 400 million embeddings for an archive that is searched a few hundred times a day. Cost matters most; one-second latency is fine. Which store?
- A. OpenSearch Service domain with HNSW
- B. Amazon S3 Vectors
- C. Aurora PostgreSQL with pgvector
- D. DynamoDB

<details><summary>Answer</summary>B. No compute to run, storage-based pricing, built for large and infrequently queried vector sets.</details>

**2.** Users search policy documents by meaning, but also by exact policy codes such as "NCB-7731". Semantic search alone misses the codes. What fixes it with the least custom code?
- A. Increase the embedding dimension
- B. Use hybrid search on OpenSearch
- C. Lower the top-K value
- D. Switch the metric from cosine to Euclidean

<details><summary>Answer</summary>B. Hybrid search combines keyword (BM25) and vector scores; exact tokens are a keyword-search strength.</details>

**3.** A team changes from a 1024-dimension embedding model to a 512-dimension one. What must they do?
- A. Update the index dimension setting in place
- B. Nothing; shorter vectors are padded automatically
- C. Create a new index with dimension 512 and re-embed and reload all documents
- D. Change only the query embeddings

<details><summary>Answer</summary>C. The dimension is fixed per index, and query and document vectors must come from the same model.</details>

**4.** An engineer creates an OpenSearch Serverless collection and has full IAM permissions, but index creation returns 403. What is the most likely cause?
- A. The collection has no standby replicas
- B. No data access policy grants the engineer's principal permissions on the index
- C. The encryption policy uses an AWS owned key
- D. The k-NN setting is disabled

<details><summary>Answer</summary>B. Serverless needs a data access policy naming the principal, in addition to IAM permissions.</details>

**5.** A filtered vector search for `category = claims` with K = 10 often returns only 2 or 3 results although many matching documents exist. Why?
- A. The filter is applied after the k-NN search (post-filtering)
- B. The distance metric is wrong
- C. The index has too many shards
- D. The embedding model is multimodal

<details><summary>Answer</summary>A. Post-filtering first takes the 10 nearest of all documents and then drops non-matching ones. Filter inside the k-NN search instead.</details>

## Save state

In [ ]:
U.save_state(vector_bucket=VBUCKET, nb_1_1b_done=True)

## Cleanup
**Run this section.** The OpenSearch collection bills by the hour until it is deleted.

Delete the OpenSearch indexes.

In [ ]:
for name in ["policy-chunks", "claim-notes", "damage-images"]:
    U.safe(os_client.indices.delete, index=name)

Delete the collection (this stops the billing).

In [ ]:
found = aoss.batch_get_collection(names=[COLL])["collectionDetails"]
if found:
    U.safe(aoss.delete_collection, id=found[0]["id"])
print("collections found:", len(found))

Delete the three policies.

In [ ]:
U.safe(aoss.delete_access_policy, name=f"{COLL}-data", type="data")
U.safe(aoss.delete_security_policy, name=f"{COLL}-net", type="network")
U.safe(aoss.delete_security_policy, name=f"{COLL}-enc", type="encryption")

Confirm that no collection is left.

In [ ]:
print(aoss.list_collections()["collectionSummaries"])

Delete the three S3 Vectors lab indexes. The vector bucket is **kept**: the Knowledge Base in 3.2b uses it.

In [ ]:
for name in ["lab-policy-1024", "lab-notes-256", "lab-images-384"]:
    U.safe(s3v.delete_index, vectorBucketName=VBUCKET, indexName=name)